# Gaussian Filtering Tuning - Result Visualization

**Purpose**: Visualize and analyze results from gaussian_filtering_tuning experiment

**Dataset**: Results from `/content/drive/MyDrive/model1/runs/gaussian_filtering_tuning/`

**Analysis**:
1. Confidence threshold optimization
2. Performance metrics comparison
3. Precision-Recall analysis
4. Best configuration identification
5. Stage1 pass rate impact

## 1. Setup: Mount Drive & Install Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
!pip install matplotlib seaborn pandas numpy -q

In [ ]:
# Imports
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## 2. Load Results Data

In [ ]:
# Path to results (update if needed)
results_path = '/content/drive/MyDrive/model1/runs/gaussian_filtering_tuning/all_results.json'

# Load JSON data
with open(results_path, 'r') as f:
    results = json.load(f)

print(f"Loaded {len(results)} result configurations")
print(f"\nFirst result sample:")
print(json.dumps(results[0], indent=2))

In [ ]:
# Convert to pandas DataFrame
df = pd.DataFrame(results)

# Extract confusion matrix values
df['TN'] = df['confusion_matrix'].apply(lambda x: x[0][0])
df['FP'] = df['confusion_matrix'].apply(lambda x: x[0][1])
df['FN'] = df['confusion_matrix'].apply(lambda x: x[1][0])
df['TP'] = df['confusion_matrix'].apply(lambda x: x[1][1])

print("\nDataFrame shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nUnique Stage1 thresholds:", sorted(df['confidence_stage1'].unique()))
print("Unique Stage2 thresholds:", sorted(df['confidence_stage2'].unique()))
print("\nSummary statistics:")
print(df[['accuracy', 'precision', 'recall', 'f1', 'specificity', 'stage1_pass_rate']].describe())

## 3. Find Best Configurations

In [ ]:
# Find best configurations by different metrics
best_configs = {
    'accuracy': df.loc[df['accuracy'].idxmax()],
    'precision': df.loc[df['precision'].idxmax()],
    'recall': df.loc[df['recall'].idxmax()],
    'f1': df.loc[df['f1'].idxmax()],
    'specificity': df.loc[df['specificity'].idxmax()]
}

print("Best Configurations by Metric:")
print("=" * 80)
for metric, config in best_configs.items():
    print(f"\n{metric.upper()}:")
    print(f"  Stage1 Confidence: {config['confidence_stage1']:.2f}")
    print(f"  Stage2 Confidence: {config['confidence_stage2']:.2f}")
    print(f"  Accuracy: {config['accuracy']:.4f}")
    print(f"  Precision: {config['precision']:.4f}")
    print(f"  Recall: {config['recall']:.4f}")
    print(f"  F1: {config['f1']:.4f}")
    print(f"  Specificity: {config['specificity']:.4f}")
    print(f"  Stage1 Pass Rate: {config['stage1_pass_rate']:.4f}")
    print(f"  Confusion Matrix: {config['confusion_matrix']}")

## 4. Heatmaps: Stage1 vs Stage2 Confidence Thresholds

In [ ]:
# Create pivot tables for heatmaps
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1']

fig, axes = plt.subplots(2, 2, figsize=(18, 16))
axes = axes.flatten()

for idx, metric in enumerate(metrics_to_plot):
    pivot = df.pivot_table(
        values=metric,
        index='confidence_stage1',
        columns='confidence_stage2',
        aggfunc='mean'
    )
    
    sns.heatmap(
        pivot,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        ax=axes[idx],
        cbar_kws={'label': metric.capitalize()},
        vmin=pivot.min().min(),
        vmax=pivot.max().max()
    )
    
    axes[idx].set_title(f'{metric.capitalize()} vs Confidence Thresholds', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Stage2 Confidence Threshold', fontsize=12)
    axes[idx].set_ylabel('Stage1 Confidence Threshold', fontsize=12)

plt.tight_layout()
plt.savefig('/content/gaussian_filtering_tuning_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: /content/gaussian_filtering_tuning_heatmaps.png")

## 5. Performance Metrics Comparison

In [ ]:
# Plot performance metrics across different Stage1 confidence thresholds
stage1_thresholds = sorted(df['confidence_stage1'].unique())

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

metrics = ['accuracy', 'precision', 'recall', 'f1']

for idx, metric in enumerate(metrics):
    for stage1_conf in stage1_thresholds:
        subset = df[df['confidence_stage1'] == stage1_conf].sort_values('confidence_stage2')
        axes[idx].plot(
            subset['confidence_stage2'],
            subset[metric],
            marker='o',
            label=f'Stage1={stage1_conf:.2f}',
            linewidth=2,
            markersize=8,
            alpha=0.7
        )
    
    axes[idx].set_xlabel('Stage2 Confidence Threshold', fontsize=12)
    axes[idx].set_ylabel(metric.capitalize(), fontsize=12)
    axes[idx].set_title(f'{metric.capitalize()} vs Stage2 Confidence', fontsize=14, fontweight='bold')
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim([0, 1])

plt.tight_layout()
plt.savefig('/content/gaussian_filtering_tuning_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: /content/gaussian_filtering_tuning_metrics.png")

## 6. Precision-Recall Tradeoff

In [ ]:
# Precision-Recall curve for different Stage1 thresholds
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

colors = plt.cm.rainbow(np.linspace(0, 1, len(stage1_thresholds)))

for stage1_conf, color in zip(stage1_thresholds, colors):
    subset = df[df['confidence_stage1'] == stage1_conf]
    ax.plot(
        subset['recall'],
        subset['precision'],
        'o-',
        label=f'Stage1={stage1_conf:.2f}',
        color=color,
        markersize=8,
        linewidth=2,
        alpha=0.7
    )

# Mark best F1 point
best_f1 = best_configs['f1']
ax.scatter(
    [best_f1['recall']],
    [best_f1['precision']],
    color='red',
    s=400,
    marker='*',
    edgecolors='black',
    linewidths=3,
    label=f'Best F1={best_f1["f1"]:.4f} (Stage1={best_f1["confidence_stage1"]:.2f}, Stage2={best_f1["confidence_stage2"]:.2f})',
    zorder=10
)

ax.set_xlabel('Recall', fontsize=14, fontweight='bold')
ax.set_ylabel('Precision', fontsize=14, fontweight='bold')
ax.set_title('Precision-Recall Tradeoff', fontsize=16, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

# Add F1 iso-lines
for f1_val in [0.5, 0.6, 0.7, 0.8, 0.9]:
    recall_range = np.linspace(0.01, 1, 100)
    precision_line = (f1_val * recall_range) / (2 * recall_range - f1_val)
    precision_line = np.clip(precision_line, 0, 1)
    ax.plot(recall_range, precision_line, '--', color='gray', alpha=0.3, linewidth=0.8)
    if precision_line[-1] < 0.98:
        ax.text(0.95, precision_line[-1], f'F1={f1_val}', fontsize=8, alpha=0.5)

plt.tight_layout()
plt.savefig('/content/gaussian_filtering_tuning_pr_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: /content/gaussian_filtering_tuning_pr_curve.png")

## 7. Stage1 Pass Rate Analysis

In [ ]:
# Analyze Stage1 pass rate impact
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

metrics = ['accuracy', 'precision', 'recall', 'f1']

for idx, metric in enumerate(metrics):
    for stage1_conf in stage1_thresholds:
        subset = df[df['confidence_stage1'] == stage1_conf]
        axes[idx].scatter(
            subset['stage1_pass_rate'],
            subset[metric],
            label=f'Stage1={stage1_conf:.2f}',
            s=100,
            alpha=0.6
        )
    
    axes[idx].set_xlabel('Stage1 Pass Rate', fontsize=12)
    axes[idx].set_ylabel(metric.capitalize(), fontsize=12)
    axes[idx].set_title(f'{metric.capitalize()} vs Stage1 Pass Rate', fontsize=14, fontweight='bold')
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim([0, 1])

plt.tight_layout()
plt.savefig('/content/gaussian_filtering_tuning_pass_rate.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: /content/gaussian_filtering_tuning_pass_rate.png")

## 8. Top Configurations Summary

In [ ]:
# Get top 5 configurations by F1 score
top_configs = df.nlargest(5, 'f1')[[
    'confidence_stage1', 'confidence_stage2', 
    'accuracy', 'precision', 'recall', 'f1', 'specificity', 'stage1_pass_rate'
]].reset_index(drop=True)

print("Top 5 Configurations by F1 Score:")
print("=" * 100)
print(top_configs.to_string(index=True))

# Visualize top 5
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

x = np.arange(len(top_configs))
width = 0.15

metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'specificity']
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']

for idx, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    offset = width * (idx - 2)
    bars = ax.bar(x + offset, top_configs[metric], width, label=metric.capitalize(), color=color, alpha=0.8)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=7, rotation=0)

ax.set_xlabel('Configuration Rank', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Top 5 Configurations Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([
    f"#{i+1}\n(S1={top_configs.loc[i, 'confidence_stage1']:.2f}, S2={top_configs.loc[i, 'confidence_stage2']:.2f})"
    for i in range(len(top_configs))
], fontsize=9)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('/content/gaussian_filtering_tuning_top5.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSaved: /content/gaussian_filtering_tuning_top5.png")

## 9. Confusion Matrix for Best Configuration

In [ ]:
# Visualize confusion matrix for best F1 configuration
best_f1_config = best_configs['f1']

fig, ax = plt.subplots(1, 1, figsize=(8, 7))

cm = np.array(best_f1_config['confusion_matrix'])
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    ax=ax,
    cbar_kws={'label': 'Count'},
    square=True,
    linewidths=2,
    linecolor='black'
)

ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_title(
    f'Confusion Matrix - Best F1 Configuration\n'
    f'(Stage1={best_f1_config["confidence_stage1"]:.2f}, Stage2={best_f1_config["confidence_stage2"]:.2f}, F1={best_f1_config["f1"]:.4f})',
    fontsize=13,
    fontweight='bold'
)
ax.set_xticklabels(['Real (0)', 'Fake (1)'])
ax.set_yticklabels(['Real (0)', 'Fake (1)'])

# Add performance metrics as text
metrics_text = (
    f"Accuracy: {best_f1_config['accuracy']:.4f}\n"
    f"Precision: {best_f1_config['precision']:.4f}\n"
    f"Recall: {best_f1_config['recall']:.4f}\n"
    f"F1: {best_f1_config['f1']:.4f}\n"
    f"Specificity: {best_f1_config['specificity']:.4f}"
)
ax.text(
    1.15, 0.5, metrics_text,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment='center',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
)

plt.tight_layout()
plt.savefig('/content/gaussian_filtering_tuning_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: /content/gaussian_filtering_tuning_confusion_matrix.png")

## 10. Save Summary Report

In [ ]:
# Create summary report
summary = f"""
Gaussian Filtering Tuning - Analysis Summary
{'='*80}

Total Configurations Tested: {len(df)}
Stage1 Thresholds Tested: {sorted(df['confidence_stage1'].unique())}
Stage2 Thresholds Tested: {sorted(df['confidence_stage2'].unique())}

BEST CONFIGURATIONS:
{'-'*80}

Best Accuracy: {best_configs['accuracy']['accuracy']:.4f}
  Stage1: {best_configs['accuracy']['confidence_stage1']:.2f}
  Stage2: {best_configs['accuracy']['confidence_stage2']:.2f}
  Confusion Matrix: {best_configs['accuracy']['confusion_matrix']}

Best Precision: {best_configs['precision']['precision']:.4f}
  Stage1: {best_configs['precision']['confidence_stage1']:.2f}
  Stage2: {best_configs['precision']['confidence_stage2']:.2f}
  Confusion Matrix: {best_configs['precision']['confusion_matrix']}

Best Recall: {best_configs['recall']['recall']:.4f}
  Stage1: {best_configs['recall']['confidence_stage1']:.2f}
  Stage2: {best_configs['recall']['confidence_stage2']:.2f}
  Confusion Matrix: {best_configs['recall']['confusion_matrix']}

Best F1: {best_configs['f1']['f1']:.4f}
  Stage1: {best_configs['f1']['confidence_stage1']:.2f}
  Stage2: {best_configs['f1']['confidence_stage2']:.2f}
  Accuracy: {best_configs['f1']['accuracy']:.4f}
  Precision: {best_configs['f1']['precision']:.4f}
  Recall: {best_configs['f1']['recall']:.4f}
  Specificity: {best_configs['f1']['specificity']:.4f}
  Stage1 Pass Rate: {best_configs['f1']['stage1_pass_rate']:.4f}
  Confusion Matrix: {best_configs['f1']['confusion_matrix']}

Best Specificity: {best_configs['specificity']['specificity']:.4f}
  Stage1: {best_configs['specificity']['confidence_stage1']:.2f}
  Stage2: {best_configs['specificity']['confidence_stage2']:.2f}
  Confusion Matrix: {best_configs['specificity']['confusion_matrix']}

OVERALL STATISTICS:
{'-'*80}

Accuracy:     Mean={df['accuracy'].mean():.4f}, Std={df['accuracy'].std():.4f}, Min={df['accuracy'].min():.4f}, Max={df['accuracy'].max():.4f}
Precision:    Mean={df['precision'].mean():.4f}, Std={df['precision'].std():.4f}, Min={df['precision'].min():.4f}, Max={df['precision'].max():.4f}
Recall:       Mean={df['recall'].mean():.4f}, Std={df['recall'].std():.4f}, Min={df['recall'].min():.4f}, Max={df['recall'].max():.4f}
F1:           Mean={df['f1'].mean():.4f}, Std={df['f1'].std():.4f}, Min={df['f1'].min():.4f}, Max={df['f1'].max():.4f}
Specificity:  Mean={df['specificity'].mean():.4f}, Std={df['specificity'].std():.4f}, Min={df['specificity'].min():.4f}, Max={df['specificity'].max():.4f}
Pass Rate:    Mean={df['stage1_pass_rate'].mean():.4f}, Std={df['stage1_pass_rate'].std():.4f}, Min={df['stage1_pass_rate'].min():.4f}, Max={df['stage1_pass_rate'].max():.4f}

TOP 5 CONFIGURATIONS:
{'-'*80}
{top_configs.to_string()}

Generated Visualizations:
{'-'*80}
1. gaussian_filtering_tuning_heatmaps.png - Performance heatmaps
2. gaussian_filtering_tuning_metrics.png - Metrics comparison
3. gaussian_filtering_tuning_pr_curve.png - Precision-Recall curve
4. gaussian_filtering_tuning_pass_rate.png - Stage1 pass rate analysis
5. gaussian_filtering_tuning_top5.png - Top 5 configurations
6. gaussian_filtering_tuning_confusion_matrix.png - Best F1 confusion matrix
"""

print(summary)

# Save to file
with open('/content/gaussian_filtering_tuning_summary.txt', 'w') as f:
    f.write(summary)

print("\nSummary saved to: /content/gaussian_filtering_tuning_summary.txt")

## Done!

**Analysis complete!**

**Generated files**:
- `gaussian_filtering_tuning_heatmaps.png`: Performance heatmaps
- `gaussian_filtering_tuning_metrics.png`: Metrics comparison
- `gaussian_filtering_tuning_pr_curve.png`: Precision-Recall curve
- `gaussian_filtering_tuning_pass_rate.png`: Stage1 pass rate analysis
- `gaussian_filtering_tuning_top5.png`: Top 5 configurations
- `gaussian_filtering_tuning_confusion_matrix.png`: Best F1 confusion matrix
- `gaussian_filtering_tuning_summary.txt`: Summary report

All files are saved in `/content/` directory